# Controlled hybrid CNN–quantum ECG experiment

This notebook compares classical and quantum heads over the same leakage-safe 12-lead CNN encoder. Every candidate uses the same raw-record manifest, grouped multilabel folds, training-only normalization, class-weighted loss, and validation macro-AUPRC.

The test stage is deliberately separate. Do not run it for every candidate.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

from ecg_experiment.config import resolve_repo_root
from ecg_experiment.hybrid_models import build_comparison_model, parameter_report

repo_root = resolve_repo_root()

## Architecture controls

`linear` is the minimal classical head. `matched-mlp` uses a classical nonlinear feature layer chosen to approximate the selected circuit's trainable parameter count. `hybrid-vqc` uses angle embedding and strongly entangling layers. `hybrid-qcnn` uses hierarchical convolution and pooling gates. Every model returns nine independent logits for the multilabel task.

In [ ]:
for model_name in ('linear', 'matched-mlp', 'hybrid-vqc', 'hybrid-qcnn'):
    model = build_comparison_model(
        model_name, 9, encoder_dim=32, n_qubits=4, quantum_depth=1
    )
    print(model_name, parameter_report(model))

## Validation-only screening

Screen 4/8 qubits and depths 1/2 with seed 43. Use a distinct output directory for each run. The command below demonstrates one candidate and does not evaluate the test set. Run `generate_hybrid_experiment_grid.py` to record the complete prespecified grid. Test angle embedding first, then angle data re-uploading only as a registered ablation. Finite-shot and depolarizing-noise runs belong after architecture selection.

In [ ]:
subprocess.run([sys.executable, str(repo_root / 'Code' / 'generate_hybrid_experiment_grid.py')], check=True)

In [ ]:
model_name = 'hybrid-vqc'
n_qubits = 4
quantum_depth = 1
seed = 43
output_dir = repo_root / 'artifacts' / 'hybrid' / model_name / f'q{n_qubits}-d{quantum_depth}' / f'seed-{seed}'
command = [
    sys.executable, str(repo_root / 'Code' / 'run_hybrid_experiment.py'),
    '--model', model_name, '--stage', 'validate',
    '--n-qubits', str(n_qubits), '--quantum-depth', str(quantum_depth),
    '--seed', str(seed), '--output-dir', str(output_dir),
]
subprocess.run(command, check=True)

## Confirmation

After screening, prespecify the finalists and repeat them for seeds 13, 23, 33, 43, and 53. Compare both a jointly trained encoder and a pretrained frozen encoder. A frozen run requires `--encoder-checkpoint` from the classical baseline. Select the winner from validation results and confidence intervals, not from test results.

## Sealed test evaluation

Replace the values below with the single prespecified winning configuration. The runner refuses to test when its configuration differs from the saved validation run.

In [ ]:
RUN_SEALED_TEST = False
if RUN_SEALED_TEST:
    test_command = command.copy()
    test_command[test_command.index('--stage') + 1] = 'test'
    subprocess.run(test_command, check=True)
else:
    print('Test set remains sealed.')